In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

## Imports

In [ ]:
import json
import torch
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    roc_auc_score,
    f1_score,
    confusion_matrix
)
from typing import Dict
import torch.nn.functional as F
import torch.nn as nn
import matplotlib.pyplot as plt
from pathlib import Path
from transformers import AutoTokenizer

## Dataset

In [ ]:
LABEL_MAP = {
    "normal": 0,
    "malicious": 1,
}

In [ ]:
class TextDataset(Dataset):
    """
    Expected JSONL format per row:
    {
        "id": "...",
        "input_text": "...",
        "target_label": "normal" | "malicious",
        "meta": {...}
    }
    """

    def __init__(self, jsonl_path: str, tokenizer, max_len: int = 512):
        self.samples = []
        self.tokenizer = tokenizer
        self.max_len = max_len

        with open(jsonl_path, "r", encoding="utf-8") as f:
            for line in f:
                obj = json.loads(line)

                text = obj["input_text"]
                label = obj["target_label"]

                y = LABEL_MAP[label]
                self.samples.append((text, y))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        text, y = self.samples[idx]

        enc = self.tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt",
        )

        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": torch.tensor(y, dtype=torch.float32),
        }

## DataLoaders

In [ ]:
def build_text_dataloaders(
    train_path: str,
    val_path: str,
    tokenizer,
    max_len: int = 512,
    batch_size_train: int = 32,
    batch_size_val: int = 64,
    num_workers: int = 0,
):
    train_ds = TextDataset(train_path, tokenizer, max_len)
    val_ds = TextDataset(val_path, tokenizer, max_len)

    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size_train,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=True,
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=batch_size_val,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True,
    )

    return train_loader, val_loader

## Metrics

In [ ]:
def compute_classification_metrics(y_true: np.ndarray, y_pred_proba: np.ndarray, threshold: float = 0.5) -> Dict[str, float]:
    """
    y_true: shape (N,) values 0/1
    y_pred_proba: shape (N,) probabilities in [0,1]
    """
    y_true = y_true.astype(int)
    y_pred = (y_pred_proba >= threshold).astype(int)

    metrics = {}
    try:
        metrics["accuracy"] = float(accuracy_score(y_true, y_pred))
        metrics["precision"] = float(precision_score(y_true, y_pred, zero_division=0))
        metrics["recall"] = float(recall_score(y_true, y_pred, zero_division=0))
        metrics["f1"] = float(f1_score(y_true, y_pred, zero_division=0))
    except Exception:
        metrics["accuracy"] = metrics["precision"] = metrics["recall"] = metrics["f1"] = 0.0

    try:
        metrics["roc_auc"] = float(roc_auc_score(y_true, y_pred_proba))
    except Exception:
        metrics["roc_auc"] = float("nan")

    # FPR at high threshold
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()
    metrics["fpr"] = float(fp / (fp + tn)) if (fp + tn) > 0 else 0.0
    metrics["tpr"] = float(tp / (tp + fn)) if (tp + fn) > 0 else 0.0

    return metrics


## Model

In [ ]:
class AttentiveBiLSTM(nn.Module):
    """
    Strong text classifier:
    Embedding -> BiLSTM -> Multi-Head Self-Attention -> Masked Mean Pool -> MLP Head

    Input:
        input_ids: (batch, seq_len)
        attention_mask: (batch, seq_len) with 1 for tokens, 0 for padding

    Output:
        logit: (batch,)  # binary logit for BCEWithLogitsLoss
    """

    def __init__(
        self,
        vocab_size: int,
        embed_dim: int = 256,
        hidden_size: int = 256,
        num_layers: int = 2,
        num_heads: int = 4,
        dropout: float = 0.3,
        pad_idx: int = 0,
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embed_dim,
            padding_idx=pad_idx
        )

        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            bidirectional=True,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )

        self.attention = nn.MultiheadAttention(
            embed_dim=hidden_size * 2,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True,
        )

        self.norm = nn.LayerNorm(hidden_size * 2)

        self.classifier = nn.Sequential(
            nn.Linear(hidden_size * 2, 256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(256, 1),
        )

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def forward(self, input_ids, attention_mask=None):
        """
        input_ids: LongTensor (batch, seq_len)
        attention_mask: LongTensor (batch, seq_len), 1 = keep, 0 = pad
        """

        # (batch, seq_len, embed_dim)
        x = self.embedding(input_ids)

        # (batch, seq_len, 2 * hidden)
        lstm_out, _ = self.lstm(x)

        # Convert to key padding mask: True = ignore
        if attention_mask is not None:
            key_padding_mask = attention_mask == 0
        else:
            key_padding_mask = None

        # Self-attention
        attn_out, _ = self.attention(
            lstm_out, lstm_out, lstm_out,
            key_padding_mask=key_padding_mask
        )

        # Masked mean pooling
        if attention_mask is not None:
            mask = attention_mask.unsqueeze(-1).float()
            attn_out = attn_out * mask
            pooled = attn_out.sum(dim=1) / mask.sum(dim=1).clamp(min=1e-6)
        else:
            pooled = attn_out.mean(dim=1)

        pooled = self.norm(pooled)

        # Final logit
        logit = self.classifier(pooled).squeeze(-1)
        return logit

## Training COnfiguration

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

MODEL_NAME = "roberta-base"

SEQ_TRAIN_PATH = "/kaggle/input/graphql-sequnce-dataset/train.jsonl"
SEQ_VAL_PATH   = "/kaggle/input/graphql-sequnce-dataset/val.jsonl"

ARTIFACT_PATH = Path("bilstm_best.pt")
METRICS_PATH  = Path("bilstm_metrics.json")

EPOCHS = 20
LR = 5e-4
WEIGHT_DECAY = 1e-2
GRAD_CLIP = 1.0

In [ ]:
def train():
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    pad_idx = tokenizer.pad_token_id
    
    TOKENIZER_DIR = Path("bilstm_tokenizer")
    TOKENIZER_DIR.mkdir(parents=True, exist_ok=True)
    tokenizer.save_pretrained(TOKENIZER_DIR)

    train_loader, val_loader = build_text_dataloaders(
        SEQ_TRAIN_PATH,
        SEQ_VAL_PATH,
        tokenizer,
        max_len=256,              # ✅ reduced (safe + faster)
        batch_size_train=32,
        batch_size_val=64,
    )

    model = AttentiveBiLSTM(
        vocab_size=tokenizer.vocab_size,
        pad_idx=pad_idx,
    ).to(DEVICE)

    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LR,
        weight_decay=WEIGHT_DECAY,
    )

    best_auc = 0.0
    history = []

    for epoch in range(1, EPOCHS + 1):
        # =========================
        # Train
        # =========================
        model.train()
        epoch_loss = 0.0

        train_pbar = tqdm(
            train_loader,
            desc=f"[BiLSTM][Epoch {epoch:02d}] Train",
            leave=False,
        )

        for batch in train_pbar:
            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            labels = batch["labels"].to(DEVICE)

            logits = model(input_ids, attention_mask)
            loss = criterion(logits, labels)

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()

            epoch_loss += loss.item()
            train_pbar.set_postfix(loss=f"{loss.item():.4f}")

        avg_train_loss = epoch_loss / len(train_loader)

        # =========================
        # Validation
        # =========================
        model.eval()
        all_logits, all_labels = [], []

        val_pbar = tqdm(
            val_loader,
            desc=f"[BiLSTM][Epoch {epoch:02d}] Val",
            leave=False,
        )

        with torch.no_grad():
            for batch in val_pbar:
                input_ids = batch["input_ids"].to(DEVICE)
                attention_mask = batch["attention_mask"].to(DEVICE)
                labels = batch["labels"].to(DEVICE)

                logits = model(input_ids, attention_mask)

                all_logits.append(logits)
                all_labels.append(labels)

        logits = torch.cat(all_logits)
        labels = torch.cat(all_labels)

        probs = torch.sigmoid(logits).cpu().numpy()
        y_true = labels.cpu().numpy()

        metrics = compute_classification_metrics(
            y_true=y_true,
            y_pred_proba=probs,
            threshold=0.5,
        )

        metrics.update({
            "epoch": epoch,
            "train_loss": avg_train_loss,
        })

        history.append(metrics)

        # =========================
        # Checkpoint
        # =========================
        if metrics["roc_auc"] > best_auc:
            best_auc = metrics["roc_auc"]
            torch.save(model.state_dict(), ARTIFACT_PATH)
            saved = True
        else:
            saved = False

        print(
            f"[BiLSTM] Epoch {epoch:02d} | "
            f"Loss={avg_train_loss:.4f} | "
            f"AUC={metrics['roc_auc']:.4f} | "
            f"P={metrics['precision']:.4f} | "
            f"R={metrics['recall']:.4f} | "
            f"F1={metrics['f1']:.4f} | "
            f"FPR={metrics['fpr']:.4f} | "
            f"Saved={saved}"
        )

    # =========================
    # Save metrics JSON
    # =========================
    METRICS_PATH.parent.mkdir(parents=True, exist_ok=True)
    with open(METRICS_PATH, "w") as f:
        json.dump(history, f, indent=2)

    print(f"Training complete. Best AUC: {best_auc:.4f}")
    print(f"Metrics saved to: {METRICS_PATH}")

## Training

In [ ]:
# print("DEVICE:", DEVICE)
# train()

In [ ]:
with open("bilstm_metrics.json", "r") as f:
    history = json.load(f)

epochs = [m["epoch"] for m in history]

## Evaluation and Visualization

In [ ]:
train_loss = [m["train_loss"] for m in history]

plt.figure()
plt.plot(epochs, train_loss, marker="o")
plt.xlabel("Epoch")
plt.ylabel("Training Loss")
plt.title("BiLSTM Training Loss")
plt.grid(True)
plt.show()

In [ ]:
auc = [m["roc_auc"] for m in history]

plt.figure()
plt.plot(epochs, auc, marker="o")
plt.xlabel("Epoch")
plt.ylabel("ROC-AUC")
plt.title("BiLSTM Validation ROC-AUC")
plt.ylim(0.95, 1.0)
plt.grid(True)
plt.show()


In [ ]:
precision = [m["precision"] for m in history]
recall = [m["recall"] for m in history]
f1 = [m["f1"] for m in history]

plt.figure()
plt.plot(epochs, precision, label="Precision")
plt.plot(epochs, recall, label="Recall")
plt.plot(epochs, f1, label="F1")
plt.xlabel("Epoch")
plt.ylabel("Score")
plt.title("Precision / Recall / F1")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
fpr = [m["fpr"] for m in history]

plt.figure()
plt.plot(epochs, fpr, marker="o")
plt.xlabel("Epoch")
plt.ylabel("False Positive Rate")
plt.title("False Positive Rate Over Epochs")
plt.yscale("log")   # IMPORTANT
plt.grid(True, which="both")
plt.show()


In [ ]:
plt.figure()
plt.scatter(fpr, recall)
plt.xlabel("False Positive Rate")
plt.ylabel("Recall")
plt.title("Recall vs False Positive Rate")
plt.grid(True)
plt.show()
